# Testing Parallel OT
OT has been quite slow thus far. To speed this up, we've implemented a version of the main deviance-guided OT function that should parallelize the work. Here, we'll test if this has identical results to the main function and how much faster it is.

## Imports

In [1]:
### Enabling autoreload ##
%load_ext autoreload
%autoreload 2

In [2]:
### Directories and Files ###
root = '../../../../../'
metadata_dir = f"{root}Data/sc_data/cell_types.tsv"
data_dir = '/Users/oliviersmeets/Desktop/University/Master Internship 1/Data Handling/Data/sc_data' # Importing was not working with relative root+'..' approach, so replace with local data
save_dir = f"{root}Generated Data/Single-to-single/sc_uot_clustering/OT/Parallelization"

In [3]:
### Imports ###
import sys
import pandas as pd
from time import perf_counter
sys.path.append(root+'Scripts')
from hicdatautils import hic_ot_optim, subset_clr_data, hic_ot_bulk_deviance, hic_ot_bulk_clr

In [4]:
### Pre-importing metadata ###
metadata_df = pd.read_csv(metadata_dir, sep="\t")
metadata_df

,cell_name,cell_type
0,arc_pair_7,T cells
1,arc_pair_18,CD4+ T cells
2,arc_pair_31,CD4+ T cells
3,arc_pair_56,T cells
4,arc_pair_57,CD8+ T cells
...,...,...
8058,arc_pair_736107,T cells
8059,arc_pair_736132,CD8+ T cells
8060,arc_pair_736144,CD8+ T cells
8061,arc_pair_736230,Monocytes


## Helper Functions

In [5]:
def generate_ot_data(
        reg_m: float,
        selection: str, 
        count: int=40, 
        seed: int=42,
        top_number: int|None=None,
        thres: float|None=None,
        thres_type: str|None=None,
        optimize: bool=True
        ) -> tuple[pd.DataFrame, float]:
    '''
        Generates pair-wise OT comparison data for the immune cell data
        using the provided arguments. Additionally, keeps track of computation
        time.

        Function is specific to this notebook.

        Parameters
        ----------
        reg_m : float
            Unbalanced parameter to use.
            Default = 10.
        selection: str,
            Type of contact selection to use.
            Options include "deviance" and "threshold".
        count : int
            The top number of cells to be used.
            Default = 20.
        seed : int
            Seed for subsetting
            Default = 42.
        top_number : int
            Top number of contacts to use based on deviance.
            Default = None.
        thres : float
            The cut-off value for setting a threshold.
            Default = None, disables threshold selection
        thres_type : str
            The kind of threshold to be set.
            Options include 'raw' and 'percentile'.
            Default = None, disables threshold selection.
        optimize : bool
            Determines whether or not to use the optimized function.
            Default = True

        Returns
        -------
        ot_results : pd.DataFrame
            Pandas dataframe with the desired results.
    '''
    # Subsetting
    clrs = subset_clr_data(data_dir, metadata_df, count, seed)
    
    # OT
        # Starting timer
    start_time = perf_counter()

        # Calculations
    if optimize:
        ot_results = hic_ot_optim(
            clrs,
            "chr1",
            selection,
            "max",
            reg_m,
            top_contacts=top_number,
            thres=thres,
            thres_type=thres_type
            )
        ot_results.to_csv(f'{save_dir}/IMMUNE_CELLS_DEVIANCE_PARALLEL_unbalanced_reg_m={reg_m}_cluster_top={top_number}_thres={thres}_count={count}_seed={seed}.csv')
    else:
        if selection == "deviance":
            ot_results = hic_ot_bulk_deviance(
                clrs,
                "chr1",
                top_number,
                "max",
                reg_m
                )
        elif selection == "threshold":
            ot_results = hic_ot_bulk_clr(
                clrs,
                clrs,
                "chr1",
                1,
                thres,
                thres_type,
                "max",
                reg_m
                )
        ot_results.to_csv(f'{save_dir}/IMMUNE_CELLS_DEVIANCE_unbalanced_reg_m={reg_m}_cluster_top={top_number}_thres={thres}_count={count}_seed={seed}.csv')
        
        # Stopping timer
    end_time = perf_counter() - start_time

    return ot_results, end_time

## Generating Data

In [ ]:
_, time_optim_dev = generate_ot_data(0.1, "deviance", 40, 42, top_number=250, optimize=True)
_, time_optim_dev = generate_ot_data(0.1, "deviance", 40, 42, top_number=250, optimize=False)
_, time_optim_thres = generate_ot_data(0.1, "threshold", 40, 42, thres=95, thres_type="percentile", optimize=True)
_, time_default_thres = generate_ot_data(0.1, "threshold", 40, 42, thres=95, thres_type="percentile", optimize=False)

/Users/oliviersmeets/Desktop/University/Master Internship 1/Data Handling/Testing/HiCOT Experimenting/Single Cell/UOT/Immune Cells/../../../../../Scripts/hicdatautils/hicgeneral.py:125: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(
  0%|          | 176/42778 [00:05<15:18, 46.40it/s] /opt/anaconda3/envs/hicotenv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
  2%|▏         | 840/42778 [00:35<28:38, 24.40it/s]

KeyboardInterrupt: 

## Loading Data and Validating Equavalence

In [ ]:
optim = pd.read_csv(f'{save_dir}/IMMUNE_CELLS_DEVIANCE_PARALLEL_unbalanced_reg_m=0.1_cluster_top=250_count=40_seed=42.csv', index_col=0)
default = pd.read_csv(f'{save_dir}/IMMUNE_CELLS_DEVIANCE_unbalanced_reg_m=0.1_cluster_top=250_count=40_seed=42.csv', index_col=0)

True
Factor by which optimized method is faster: 3.84x


Outputs are identical